In [1]:
import pandas as pd
import scipy as sc
import numpy as np
import xgboost as xgb
import sklearn as sk
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df_22 = pd.read_csv(r"D:\F1_ML\DATA\df_2022.csv")
df_23 = pd.read_csv(r"D:\F1_ML\DATA\df_2023.csv")
df_24 = pd.read_csv(r"D:\F1_ML\DATA\df_2024.csv")
df_25 = pd.read_csv(r"D:\F1_ML\DATA\df_2025.csv")
df_26 = pd.read_csv(r"D:\F1_ML\DATA\df_2026.csv")


In [3]:
print(df_22.shape,
df_23.shape,
df_24.shape,
df_25.shape,
df_26.shape)

(1160, 26) (1160, 26) (1238, 26) (1279, 26) (176, 26)


In [4]:
df = pd.concat([df_22, df_23, df_24, df_25, df_26])

In [5]:
df = df.drop(columns=['HeadshotUrl', 'TeamColor', 'BroadcastName', 'FirstName', 'LastName', 'DriverNumber', 'Abbreviation', 'TeamId', 'CountryCode'])

In [ ]:

quali_times = (
    df[df['form_of_race'] == 'Qualifying']
    [['year', 'gp', 'DriverId', 'Q1', 'Q2', 'Q3', 'Position']]
    .rename(columns={'Position': 'quali_position'})
)

races = df[df['form_of_race'] == 'Race'].drop(columns=['Q1', 'Q2', 'Q3']).copy()

df_train = races.merge(
    quali_times,
    on=['year', 'gp', 'DriverId'],
    how='left'
)


Wierszy wyścigowych: 2064
Po merge:            2064
NaN w Q1: 25
NaN w Q2: 534
NaN w Q3: 1059


In [23]:
# n_corners — oficjalne dane FIA (różne źródła czasem różnią się o ±1)
# overtaking_difficulty — subiektywna ocena 1-5
#   1 = bardzo łatwo wyprzedzać (Monza, Spa)
#   5 = praktycznie niemożliwe (Monaco, Singapore)
circuit_features = {
    'Bahrain Grand Prix':          {'n_corners': 15, 'overtaking_difficulty': 2},
    'Saudi Arabian Grand Prix':    {'n_corners': 27, 'overtaking_difficulty': 4},  
    'Australian Grand Prix':       {'n_corners': 14, 'overtaking_difficulty': 3},  
    'Japanese Grand Prix':         {'n_corners': 18, 'overtaking_difficulty': 3},  
    'Chinese Grand Prix':          {'n_corners': 16, 'overtaking_difficulty': 2},  
    'Miami Grand Prix':            {'n_corners': 19, 'overtaking_difficulty': 4},
    'Emilia Romagna Grand Prix':   {'n_corners': 19, 'overtaking_difficulty': 4},  
    'Monaco Grand Prix':           {'n_corners': 19, 'overtaking_difficulty': 5},
    'Canadian Grand Prix':         {'n_corners': 14, 'overtaking_difficulty': 2},  
    'Spanish Grand Prix':          {'n_corners': 14, 'overtaking_difficulty': 3},  
    'Austrian Grand Prix':         {'n_corners': 10, 'overtaking_difficulty': 2},  
    'British Grand Prix':          {'n_corners': 18, 'overtaking_difficulty': 2},  
    'Hungarian Grand Prix':        {'n_corners': 14, 'overtaking_difficulty': 4},  
    'Belgian Grand Prix':          {'n_corners': 19, 'overtaking_difficulty': 1},  
    'Dutch Grand Prix':            {'n_corners': 14, 'overtaking_difficulty': 4},  
    'Italian Grand Prix':          {'n_corners': 11, 'overtaking_difficulty': 1},  
    'Azerbaijan Grand Prix':       {'n_corners': 20, 'overtaking_difficulty': 3},  
    'Singapore Grand Prix':        {'n_corners': 19, 'overtaking_difficulty': 5},  
    'United States Grand Prix':    {'n_corners': 20, 'overtaking_difficulty': 2},  
    'Mexico City Grand Prix':      {'n_corners': 17, 'overtaking_difficulty': 3},
    'São Paulo Grand Prix':        {'n_corners': 15, 'overtaking_difficulty': 2},  
    'Las Vegas Grand Prix':        {'n_corners': 17, 'overtaking_difficulty': 3},
    'Qatar Grand Prix':            {'n_corners': 16, 'overtaking_difficulty': 3},  
    'Abu Dhabi Grand Prix':        {'n_corners': 16, 'overtaking_difficulty': 3},  
    'French Grand Prix':           {'n_corners': 15, 'overtaking_difficulty': 3},  
}


circuit_df = pd.DataFrame.from_dict(circuit_features, orient='index').reset_index()
circuit_df = circuit_df.rename(columns={'index': 'gp'})


df_train = df_train.merge(circuit_df, on='gp', how='left')

